<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/mesoscope_sequence_area.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mesoscope sequence-mismatch: an area dissociation (VISl vs VISp)

The Neuropixels sequence result (Result 4) is a robust positive prediction-error signal
(DvI ≈ +0.21). Does it appear in mesoscope 2-photon? The **pooled** mesoscope answer is a weak
+0.04 — but that number is misleading. Broken down by cortical area, the signal is carried
almost entirely by the **higher-order lateral area (VISl)**, not primary visual cortex (VISp):

- **VISl: DvI = +0.18** (95% CI [+0.16, +0.21], positive in 13/16 sessions)
- **VISp: DvI = −0.08** (95% CI [−0.11, −0.05], positive in only 7/16 sessions)
- Paired within-session: **VISl > VISp in 14/16 sessions** (Wilcoxon p = 0.001)

VISl matches the Neuropixels magnitude; VISp shows no sequence-PE (if anything slight
suppression). This is consistent with predictive-processing accounts in which deviance/error
signals are more prominent in higher-order cortex. It also **resolves the apparent cross-scale
attenuation**: the sequence signal is not weak in 2-photon per se — it is weak *where V1 is
oversampled*, which the whole-population pool is.

Somatic ROIs only (`is_soma`); DvI = (R_deviant90 − R_control90)/(|·|+|·|) using the equiprobable
Control block 2 as the matched control, response window 0–1 s (calcium-appropriate).

In [ ]:
import sys, subprocess
try:
    import remfile, h5py  # noqa
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","remfile","h5py","requests","pandas","numpy","matplotlib","scipy"],check=True)
import numpy as np, pandas as pd, re, h5py, remfile, requests
from scipy import stats as ss
QUICK = True   # True: 4 sessions for a fast Colab pass; False: all 16

def s3(aid, ds="001768"):
    return requests.get(f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/draft/assets/{aid}/download/",allow_redirects=False,timeout=60).headers["Location"]
def resolve_asset(subject, ses_substr, ds="001768"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/draft/assets/"
    r=requests.get(u,params={"path":f"sub-{subject}/"},timeout=30).json()
    hits=[a for a in r["results"] if ses_substr in a["path"]]
    if len(hits)!=1: raise LookupError(f"{subject}/{ses_substr}: {len(hits)} matches")
    return hits[0]["asset_id"]
def dec(a): return np.array([x.decode() if isinstance(x,bytes) else x for x in a])

In [ ]:
SESSIONS = [
 ("843001","2026-04-02"),
 ("832700","2026-02-07"),
 ("843000","2026-03-04"),
 ("839909","2026-03-06"),
 ("839909","2026-03-19"),
 ("843000","2026-03-03"),
 ("837568","2026-02-16"),
 ("832700","2026-02-06"),
 ("837568","2026-02-13"),
 ("843001","2026-04-01"),
 ("845342","2026-04-01"),
 ("846289","2026-04-21"),
 ("842971","2026-04-22"),
 ("846289","2026-04-20"),
 ("845342","2026-03-31"),
 ("842971","2026-04-18"),
]
if QUICK: SESSIONS = [SESSIONS[0],SESSIONS[6],SESSIONS[2],SESSIONS[12]]  # 2 VISl-pos, 2 both-neg
def extract(aid, subj):
    fh=h5py.File(remfile.File(s3(aid)),"r")
    g=fh["intervals"]["Sequence mismatch block_presentations"]; TT=dec(g["TrialType"][:]); ts=g["start_time"][:]
    tis=dec(g["TrialInSequence"][:]).astype(float)
    gc=fh["intervals"]["Control block 2_presentations"]; cori=dec(gc["Orientation"][:]).astype(float); cts=gc["start_time"][:]
    ctf=dec(gc["TemporalFrequency"][:]).astype(float) if "TemporalFrequency" in gc else np.full(len(cts),2.0)
    dev=ts[(TT=="orientation_90")&(tis==3)]; ctrl=cts[(np.abs(np.degrees(cori)-90)<5)&(np.abs(ctf-2)<0.5)]
    op=fh["general"]["optophysiology"]; rows=[]
    for pl in [k for k in fh["processing"].keys() if k.startswith("VIS")]:
        loc=op[pl]["location"][()]; loc=loc.decode() if isinstance(loc,bytes) else loc
        m=re.search(r"Structure:\s*(\w+)\s+Depth:\s*(\d+)",loc); area=m.group(1); depth=int(m.group(2))
        pr=fh["processing"][pl]; D=pr["dff_timeseries"]["dff_timeseries"]["data"][:]; tt=pr["dff_timeseries"]["dff_timeseries"]["timestamps"][:]
        rt=pr["image_segmentation"]["roi_table"]; is_soma=rt["is_soma"][:].astype(bool) if "is_soma" in rt else np.ones(D.shape[1],bool)
        def wr(on,rw=(0.0,1.0),bw=(-0.5,-0.05)):
            R=np.full((len(on),D.shape[1]),np.nan)
            for i,o in enumerate(on):
                l0=np.searchsorted(tt,o+rw[0]);h0=np.searchsorted(tt,o+rw[1]);lb=np.searchsorted(tt,o+bw[0]);hb=np.searchsorted(tt,o+bw[1])
                if h0>l0 and hb>lb: R[i]=np.nanmean(D[l0:h0],0)-np.nanmean(D[lb:hb],0)
            return np.nanmean(R,0)
        rd=wr(dev); rc=wr(ctrl); mk=is_soma&~(np.isnan(rd)|np.isnan(rc))
        dvi=(rd[mk]-rc[mk])/(np.abs(rd[mk])+np.abs(rc[mk])+1e-9)
        for v in dvi: rows.append((subj,area,depth,v))
    fh.close(); return rows

In [ ]:
rows=[]
for si,(subj,ses) in enumerate(SESSIONS):
    r=extract(resolve_asset(subj,ses),subj)
    rows+=[(*x,si) for x in r]
    print(f"  {si+1}/{len(SESSIONS)} {subj}: {len(r)} somatic ROIs")
DM=pd.DataFrame(rows,columns=["subject","area","depth","dvi","session"])
sm=DM.groupby(["session","area"]).dvi.median().unstack()
for area in ["VISl","VISp"]:
    print(f"{area}: DvI={DM[DM.area==area].dvi.median():+.3f}, {(sm[area]>0).sum()}/{len(sm)} sessions positive")
paired=sm.dropna(); w=ss.wilcoxon(paired["VISl"],paired["VISp"])
print(f"VISl>VISp in {(paired['VISl']>paired['VISp']).sum()}/{len(paired)} sessions, Wilcoxon p={w.pvalue:.4f}")

In [ ]:
import matplotlib.pyplot as plt
acol={"VISl":"#c0392b","VISp":"#1b4079"}
fig,(a1,a2)=plt.subplots(1,2,figsize=(9,4)); fig.subplots_adjust(wspace=0.4,top=0.85,bottom=0.15,left=0.1,right=0.97)
for y,ar in [(1,"VISl"),(0,"VISp")]:
    d=DM[DM.area==ar].dvi; a1.plot(d.median(),y,"o",color=acol[ar],ms=11)
    a1.text(d.median(),y+0.2,f"{ar} {d.median():+.2f}",ha="center",fontweight="bold",color=acol[ar],fontsize=10)
a1.axvline(0,color="k",ls=":",lw=0.8); a1.set_ylim(-0.5,1.6); a1.set_yticks([]); a1.set_xlabel("sequence DvI (90 deg)")
a1.set_title("Sequence-PE in VISl, not VISp",fontsize=9)
for _,rr in sm.iterrows(): a2.plot([0,1],[rr["VISp"],rr["VISl"]],"-",color="0.7",lw=0.8)
a2.scatter([0]*len(sm),sm["VISp"],color=acol["VISp"],s=30); a2.scatter([1]*len(sm),sm["VISl"],color=acol["VISl"],s=30)
a2.axhline(0,color="k",ls=":",lw=0.8); a2.set_xticks([0,1]); a2.set_xticklabels(["VISp","VISl"]); a2.set_ylabel("per-session median DvI")
a2.set_title("Paired VISl > VISp",fontsize=9)
for sp in ["top","right"]:
    a1.spines[sp].set_visible(False); a2.spines[sp].set_visible(False)
plt.show()

### Takeaway

The mesoscope sequence prediction-error signal is **area-specific**: robustly positive in the
higher-order lateral area (VISl, matching the Neuropixels magnitude), absent/slightly negative in
primary V1 (VISp). The pooled +0.04 is an average over a VISp-dominated population and
under-represents the real effect. This is a within-modality analogue of the predictive-coding
prediction that error signals concentrate in higher-order cortex, and it reframes the
"imaging attenuates the signal" story: the signal survives in 2-photon *where higher-order areas
are recorded*.

The per-subject spread that remains (some sessions negative in both areas) is a magnitude
modulation on top of the consistent area ordering — a candidate for a behavioral-state analysis
as more data accrues.